## **Text Summarization.AI**
I implemented an abstractive text summarization system using the T5-small pretrained model from Hugging Face’s Transformers library. The goal was to generate concise and meaningful summaries from longer input texts.
To achieve this, I fine-tuned the T5-small model on a custom dataset containing 10,000 text-summary pairs, with 8,000 examples for training and 2,000 for testing.

In [ ]:
!pip install datasets

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from datasets import load_dataset

from transformers import T5ForConditionalGeneration, T5Tokenizer, Trainer, TrainingArguments

# **Load Dataset**

In [ ]:
# Load the dataset
summarize_df = pd.read_csv("summarize_ds.csv", on_bad_lines='skip', engine='python')
summarize_df.head()

## **Data Pre-processing**

In [ ]:
# Drop the column id and modify the columns names
if 'id' in summarize_df.columns:
  summarize_df = summarize_df.drop('id', axis=1)

summarize_df = summarize_df.rename(columns={'article': 'Input_text', 'highlights': 'Summary'})

summarize_df.head()

In [ ]:
# Check if there is missing values
summarize_df.isnull().sum()

In [ ]:
summarize_df.shape

In [ ]:
summarize_df = summarize_df.sample(n=10000, random_state=42).reset_index(drop=True)
summarize_df.shape

In [ ]:
import re

# Create a method clean_text to remove all special charachters
def clean_text(text):
    text = re.sub(r'\r\n', ' ', text)  # Remove carriage returns and line breaks
    text = re.sub(r'\s+', ' ', text)  # Remove extra spaces
    text = re.sub(r'<.*?>', '', text)  # Remove any XML tags
    text = text.strip().lower()  # Strip and convert to lower case
    return text

# Apply cleaning to both Input_text and Summary in the DataFrame
summarize_df['Input_text'] = summarize_df['Input_text'].apply(clean_text)
summarize_df['Summary'] = summarize_df['Summary'].apply(clean_text)

In [ ]:
# Split data into traing and test sets

train_data, test_data = train_test_split(summarize_df, test_size=0.2, random_state=42)

print(train_data.shape)
print(test_data.shape)

# Convert from dataframe to Dataset format
train_data = Dataset.from_pandas(train_data)
test_data = Dataset.from_pandas(test_data)

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [ ]:
# Print the max length of the input text and summary
input_max_len = max(len(tokenizer.encode(example)) for example in train_data['Input_text'])
output_max_len = max(len(tokenizer.encode(example)) for example in train_data['Summary'])

input_max_len, output_max_len

## **Tokenization**

In [ ]:
# Preprocessing function for tokenization
def preprocess_function(text):
    # Tokenize the Input_text and Summary
    inputs = tokenizer(text["Input_text"], padding="max_length", truncation=True, max_length=512)
    targets = tokenizer(text["Summary"], padding="max_length", truncation=True, max_length=200)
    inputs["labels"] = targets["input_ids"]
    return inputs

# Apply the preprocessing
train_dataset = train_data.map(preprocess_function, batched=True)
test_dataset = test_data.map(preprocess_function, batched=True )

In [ ]:
#Activate the code if you want to create folder in your google drive
""" from google.colab import drive
drive.mount('/content/drive')

import os
result_dir = '/content/drive/MyDrive/results1'
model_dir = '/content/drive/MyDrive/Text_Summarizer'

# Create dir if not exist
os.makedirs(result_dir , exist_ok=True)
os.makedirs(model_dir , exist_ok=True) """

## **Fine-tune the T5-small model**

In [ ]:
# Model
model = T5ForConditionalGeneration.from_pretrained("t5-small")



data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results1",         # output directory for checkpoints
    num_train_epochs=6,              # number of training epochs
    per_device_train_batch_size=8,   # batch size per device during training
    per_device_eval_batch_size=8,    # batch size for evaluation
    warmup_steps=500,                # number of warmup steps for learning rate scheduler
    weight_decay=0.01,               # strength of weight decay
    logging_dir="./logs",            # directory for storing logs
    logging_steps=1000,                # how often to log training info
    save_steps=500,                  # how often to save a model checkpoint
    eval_steps=50,                   # how often to run evaluation
    eval_strategy="epoch",
    learning_rate=5e-5,

)

# Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,

    tokenizer=tokenizer,
)


# Train the model
trainer.train()

In [ ]:
# Save the models to google drive
# (Activate the code if you want)
""" model.save_pretrained(model_dir)
trainer.save_model(model_dir)
tokenizer.save_pretrained(model_dir) """

In [ ]:
# Load saved models from google drive (not neccessary)
"""loaded_model = T5ForConditionalGeneration.from_pretrained(model_dir)
loaded_tokenizer = T5Tokenizer.from_pretrained(model_dir)"""

In [ ]:
# Ensure the model is on the correct device (GPU if available)
device = model.device  # Get the device the model is on

def summarize_text(Input_text):
    dialogue = clean_text(Input_text)  # Assuming clean_text is defined
    inputs = tokenizer(Input_text, return_tensors="pt", truncation=True, padding="max_length", max_length=512)

    # Move input tensors to the same device as the model
    inputs = {key: value.to(device) for key, value in inputs.items()}

    # Generate summary
    outputs = model.generate(inputs["input_ids"], max_length=150,  num_beams=4, early_stopping=True)

    # Decode the generated summary and [0] get the the result out from the list
    summary = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    return summary

In [ ]:
# Test the model by input a Text

input_text1 = """

Lionel Andrés "Leo" Messi born 24 June 1987) is an Argentine professional
footballer who plays as a forward for and captains both Major League Soccer
club Inter Miami and the Argentina national team. Widely regarded as one of
the greatest players of all time, Messi set numerous records for individual
accolades won throughout his professional footballing career such as eight
Ballon d'Or awards and eight times being named the world's best player by FIFA.
He is the most decorated player in the history of professional football having
won 45 team trophies,[note 3] including twelve Big Five league titles,
four UEFA Champions Leagues, two Copa Américas, and one FIFA World Cup.

"""

summary = summarize_text(input_text1)
print(f"The Summary of the input_text1 is:\n {summary}")

In [ ]:
input_text2 = """

Fast Five \r(also known as Fast & Furious 5) is a 2011 action film directed by
Justin Lin and written by Chris Morgan. It is the sequel to Fast & Furious (2009)
and the fifth installment in the Fast & Furious franchise. The film stars Vin Diesel
as Dominic Toretto and \nPaul Walker as Brian O'Conner, alongside Jordana Brewster,
Tyrese Gibson, Gal Gadot, Chris "Ludacris" Bridges, Matt Schulze, Sung Kang and Dwayne Johnson.
In the film, Dom and Brian, \nalong with Dom's \nsister Mia (Brewster) plan a heist to steal
$100 million from corrupt businessman \rHernan Reyes (Joaquim de Almeida) while being pursued
for arrest by U.S. Diplomatic Security Service (DSS) agent Luke Hobbs (Johnson).
While developing Fast Five, Universal Pictures deliberately departed from the
street racing theme prevalent in previous films in the series, to transform the
franchise into a heist action series involving cars.

"""

summary = summarize_text(input_text2)
print(f"The Summary of the input_text2 is:\n {summary}")